In [134]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from matplotlib import cm
from matplotlib.colors import Normalize
from collections import defaultdict

import yaml
import h5py
import tqdm
# from larndsim.fee import digitize


In [4]:
def unique_channel_id(d):
    return ((d['io_group'].astype(int)*1000+d['io_channel'].astype(int))*1000 \
            + d['chip_id'].astype(int))*100 + d['channel_id'].astype(int)

def unique_to_channel_id(unique):
    return unique % 100

def unique_to_chip_id(unique):
    return (unique// 100) % 1000

def unique_to_io_channel(unique):
    return(unique//(100*1000)) % 1000

def unique_to_tiles(unique):
    return ( (unique_to_io_channel(unique)-1) // 4) + 1

def unique_to_io_group(unique):
    return(unique // (100*1000*1000)) % 1000



In [50]:
_default_geometry_yaml = 'multi_tile_layout-2.3.16.yaml'
_default_geometry_yaml_mod2 = 'larndsim/pixel_layouts/multi_tile_layout-2.5.16.yaml'

def _default_pxy():
    return (0., 0.)


def _rotate_pixel(pixel_pos, tile_orientation):
    return pixel_pos[0]*tile_orientation[2], pixel_pos[1]*tile_orientation[1]


cmap = cm.viridis_r
pixel_pitch = 1

geometry_yaml = _default_geometry_yaml
with open(geometry_yaml) as fi:
    geo = yaml.full_load(fi)


geometry_yaml_mod2 = _default_geometry_yaml_mod2
with open(geometry_yaml_mod2) as fi2:
    geo_mod2 = yaml.full_load(fi2)

# Module 1, 3, 4 layout
pixel_pitch = geo['pixel_pitch']

chip_channel_to_position = geo['chip_channel_to_position']
tile_orientations = geo['tile_orientations']
tile_positions = geo['tile_positions']
tpc_centers = geo['tpc_centers']
tile_indeces = geo['tile_indeces']
xs = np.array(list(chip_channel_to_position.values()))[
    :, 0] * pixel_pitch
ys = np.array(list(chip_channel_to_position.values()))[
    :, 1] * pixel_pitch
x_size = max(xs)-min(xs)+pixel_pitch
y_size = max(ys)-min(ys)+pixel_pitch

tile_geometry = defaultdict(int)
io_group_io_channel_to_tile = {}
geometry = defaultdict(_default_pxy)

for tile in geo['tile_chip_to_io']:
    tile_orientation = tile_orientations[tile]
    tile_geometry[tile] = tile_positions[tile], tile_orientations[tile]
    for chip in geo['tile_chip_to_io'][tile]:
        io_group_io_channel = geo['tile_chip_to_io'][tile][chip]
        io_group = io_group_io_channel//1000
        io_channel = io_group_io_channel % 1000
        io_group_io_channel_to_tile[(
            io_group, io_channel)] = tile

    for chip_channel in geo['chip_channel_to_position']:
        chip = chip_channel // 1000
        channel = chip_channel % 1000
        try:
            io_group_io_channel = geo['tile_chip_to_io'][tile][chip]
        except KeyError:
            print("Chip %i on tile %i not present in network" %
                  (chip, tile))
            continue

        io_group = io_group_io_channel // 1000
        io_channel = io_group_io_channel % 1000
        x = chip_channel_to_position[chip_channel][0] * \
            pixel_pitch + pixel_pitch / 2 - x_size / 2
        y = chip_channel_to_position[chip_channel][1] * \
            pixel_pitch + pixel_pitch / 2 - y_size / 2

        x, y = _rotate_pixel((x, y), tile_orientation)
        x += tile_positions[tile][2] + \
            tpc_centers[tile_indeces[tile][0]][0]
        y += tile_positions[tile][1] + \
            tpc_centers[tile_indeces[tile][0]][1]

        geometry[(io_group, io_group_io_channel_to_tile[(
            io_group, io_channel)], chip, channel)] = x, y

xmin = min(np.array(list(geometry.values()))[:, 0])-pixel_pitch/2
xmax = max(np.array(list(geometry.values()))[:, 0])+pixel_pitch/2
ymin = min(np.array(list(geometry.values()))[:, 1])-pixel_pitch/2
ymax = max(np.array(list(geometry.values()))[:, 1])+pixel_pitch/2

tile_vertical_lines = np.linspace(xmin, xmax, 3)
tile_horizontal_lines = np.linspace(ymin, ymax, 5)
chip_vertical_lines = np.linspace(xmin, xmax, 21)
chip_horizontal_lines = np.linspace(ymin, ymax, 41)

nonrouted_v2a_channels = [6, 7, 8, 9, 22,
                          23, 24, 25, 38, 39, 40, 54, 55, 56, 57]
routed_v2a_channels = [i for i in range(
    64) if i not in nonrouted_v2a_channels]

# Module 2 layout
pixel_pitch_mod2 = geo_mod2['pixel_pitch']

chip_channel_to_position_mod2 = geo_mod2['chip_channel_to_position']
tile_orientations_mod2 = geo_mod2['tile_orientations']
tile_positions_mod2 = geo_mod2['tile_positions']
tpc_centers_mod2 = geo['tpc_centers']
tile_indeces_mod2 = geo_mod2['tile_indeces']
xs_mod2 = np.array(list(chip_channel_to_position_mod2.values()))[
    :, 0] * pixel_pitch_mod2
ys_mod2 = np.array(list(chip_channel_to_position_mod2.values()))[
    :, 1] * pixel_pitch_mod2
x_size_mod2 = max(xs_mod2)-min(xs_mod2)+pixel_pitch_mod2
y_size_mod2 = max(ys_mod2)-min(ys_mod2)+pixel_pitch_mod2

tile_geometry_mod2 = defaultdict(int)
io_group_io_channel_to_tile_mod2 = {}
geometry_mod2 = defaultdict(_default_pxy)

for tile in geo_mod2['tile_chip_to_io']:
    tile_orientation_mod2 = tile_orientations_mod2[tile]
    tile_geometry_mod2[tile] = tile_positions_mod2[tile], tile_orientations_mod2[tile]
    for chip in geo_mod2['tile_chip_to_io'][tile]:
        io_group_io_channel = geo_mod2['tile_chip_to_io'][tile][chip]
        io_group = io_group_io_channel//1000
        io_channel = io_group_io_channel % 1000
        io_group_io_channel_to_tile_mod2[(
            io_group, io_channel)] = tile

    for chip_channel in geo_mod2['chip_channel_to_position']:
        chip = chip_channel // 1000
        channel = chip_channel % 1000
        try:
            io_group_io_channel = geo_mod2['tile_chip_to_io'][tile][chip]
        except KeyError:
            print("Chip %i on tile %i not present in Module 2 network" %
                  (chip, tile))
            continue

        io_group = io_group_io_channel // 1000
        io_channel = io_group_io_channel % 1000
        x = chip_channel_to_position_mod2[chip_channel][0] * \
            pixel_pitch_mod2 + pixel_pitch_mod2 / 2 - x_size_mod2 / 2
        y = chip_channel_to_position_mod2[chip_channel][1] * \
            pixel_pitch_mod2 + pixel_pitch_mod2 / 2 - y_size_mod2 / 2

        x, y = _rotate_pixel((x, y), tile_orientation_mod2)
        x += tile_positions_mod2[tile][2] + \
            tpc_centers_mod2[tile_indeces_mod2[tile][0]][0]
        y += tile_positions_mod2[tile][1] + \
            tpc_centers_mod2[tile_indeces_mod2[tile][0]][1]

        geometry_mod2[(io_group, io_group_io_channel_to_tile_mod2[(
            io_group, io_channel)], chip, channel)] = x, y

xmin_mod2 = min(np.array(list(geometry_mod2.values()))
                [:, 0])-pixel_pitch_mod2/2
xmax_mod2 = max(np.array(list(geometry_mod2.values()))
                [:, 0])+pixel_pitch_mod2/2
ymin_mod2 = min(np.array(list(geometry_mod2.values()))
                [:, 1])-pixel_pitch_mod2/2
ymax_mod2 = max(np.array(list(geometry_mod2.values()))
                [:, 1])+pixel_pitch_mod2/2

# Plot metrics


In [125]:
def unique_id_to_pixel_id(un, is_mod2=False):
    io_group = unique_to_io_group(un)
    tile = unique_to_io_channel(un)
    chip_id = unique_to_chip_id(un)
    channel_id = unique_to_channel_id(un)

    pitch = pixel_pitch_mod2 if is_mod2 else pixel_pitch

    gg = geometry_mod2 if is_mod2 else geometry

    x, y = gg[(2 - (io_group % 2), tile + 8 * (1 - (io_group % 2)), chip_id, channel_id)]

    x_min = xmin_mod2 if is_mod2 else xmin
    x_max = xmax_mod2 if is_mod2 else xmax
    
    y_min = ymin_mod2 if is_mod2 else ymin
    y_max = ymax_mod2 if is_mod2 else ymax

    x_int = (x - x_min) / pitch - 0.5
    y_int = (y - y_min) / pitch - 0.5

    if abs(round(x_int) - x_int) > 0.05:
        print(is_mod2)
        print('ERROR X: ', un, ' - ', round(x_int), ' vs ', x_int)
    if abs(round(y_int) - y_int) > 0.05:
        print(is_mod2)
        print('ERROR Y: ', un, ' - ', round(y_int), ' vs ', y_int)
        
    if abs(round((x_max - x_min)/pitch) - (x_max - x_min)/pitch) > 0.05:
        print('ERROR STEP X: ', un)
    if abs(round((y_max - y_min)/pitch) - (y_max - y_min)/pitch) > 0.05:
        print('ERROR STEP Y: ', un)

    npix_x = round((x_max - x_min)/pitch)
    npix_y = round((y_max - y_min)/pitch)
    
    return round(x_int) + (round(y_int) + npix_y * (1 - (io_group % 2))) * npix_x
    

In [133]:
# LArPix-v2a anodes

pixelid_to_uniqueid = dict()
uniqueid_to_pixelid = dict()
nonrouted_v2a_channels = [6, 7, 8, 9, 22, 23, 24, 25, 38, 39, 40, 54, 55, 56, 57]

for io_group in tqdm.tqdm(range(1, 3)):
    for tile in range(1, 9):
        for chip_id in range(11, 111):
            for channel_id in range(64):
                if io_group in [1, 2, 3, 4, 7, 8] and channel_id in nonrouted_v2a_channels:
                    continue
                unique_id = ((io_group*1000+tile)*1000 + chip_id)*100 + channel_id
                pixel_id = unique_id_to_pixel_id(unique_id, is_mod2=False)

                if pixel_id in pixelid_to_uniqueid.keys():
                    print('DUPLICATE PIXEL ID')
                    print('pixel_id: ', pixel_id)
                    print('channel: ', (io_group, tile, chip_id, channel_id))
                if channel_id in uniqueid_to_pixelid.keys():
                    print('DUPLICATE UNIQUE ID')
                pixelid_to_uniqueid[pixel_id] = unique_id
                uniqueid_to_pixelid[unique_id] = pixel_id

with open("pixelid_to_uniqueid_v2a.json", "w") as f:
    json.dump(pixelid_to_uniqueid, f)
with open("uniqueid_to_pixelid_v2a.json", "w") as f:
    json.dump(uniqueid_to_pixelid, f)


100%|██████████| 2/2 [00:01<00:00,  1.69it/s]


In [137]:
# LArPix-v2b anodes

pixelid_to_uniqueid = dict()
uniqueid_to_pixelid = dict()

for io_group in tqdm.tqdm(range(1, 3)):
    for tile in range(1, 9):
        for chip_id in range(11, 111):
            for channel_id in range(64):
                unique_id = ((io_group*1000+tile)*1000 + chip_id)*100 + channel_id
                pixel_id = unique_id_to_pixel_id(unique_id, is_mod2=True)

                if pixel_id in pixelid_to_uniqueid.keys():
                    print('DUPLICATE PIXEL ID')
                    print('pixel_id: ', pixel_id)
                    print('channel: ', (io_group, tile, chip_id, channel_id))
                if channel_id in uniqueid_to_pixelid.keys():
                    print('DUPLICATE UNIQUE ID')
                pixelid_to_uniqueid[pixel_id] = unique_id
                uniqueid_to_pixelid[unique_id] = pixel_id

with open("pixelid_to_uniqueid_v2b.json", "w") as f:
    json.dump(pixelid_to_uniqueid, f)
with open("uniqueid_to_pixelid_v2b.json", "w") as f:
    json.dump(uniqueid_to_pixelid, f)


[]